# AI Functions in Databricks SQL — 101

**What are AI Functions?**
Built-in SQL functions backed by state-of-the-art AI models. No endpoint to deploy, no prompt plumbing, no infrastructure — you write ordinary SQL and Databricks runs the model for you, serverless.

```sql
SELECT ai_analyze_sentiment('The biryani was amazing!')   -- returns: positive
```

**Why they matter**
* **SQL is all you need** — anyone who can write a query can now translate, classify, extract and summarize text. No Python, no ML pipeline, no model training.
* **They compose like any SQL** — use them in `SELECT`s, `CREATE TABLE AS`, views, joins and `GROUP BY`s. A model call becomes just another expression in your query.
* **Governed by default** — they run under Unity Catalog, so permissions, lineage and audit trails apply like any other SQL workload.
* **Serverless & pay-per-use** — billed per token by DBUs; no clusters to size, no endpoints to babysit. Small samples cost cents (see the cost section near the end).

**What you'll learn in this notebook**
1. The landscape — what functions exist and which family each belongs to (next cell)
2. Live examples of the **9 GA text functions** — definition, signature, parameters and one runnable example each
3. The advanced tier: `ai_query` (GA, pattern only), then the **Beta & preview functions at the end** — runnable only where enabled
4. Cost tracking — where to see what AI functions spent
5. A recap that maps each function to a real pipeline step

**How to run this notebook**
* Use a **SQL warehouse or serverless compute** (this workspace auto-selects serverless — nothing to configure).
* Every AI function call invokes a model. The examples below call them on **single literal strings**, so the whole notebook costs a few cents.
* Cells are independent — run any of them in any order; nothing to set up first.
* Runtime: ~2 minutes.

**Companion notebook:** `02_reviews_pipeline` puts these functions to work in a full medallion pipeline — 25 multilingual food-delivery reviews flowing from bronze to governed gold tables, with data-quality rules, PII masking, a 6-function enrichment query, cost-capped AI drafting and Unity Catalog governance. Run this one first for the tour; that one for the real thing.

Docs: [AI functions in Databricks](https://docs.databricks.com/aws/en/large-language-models/ai-functions)

## The AI Functions landscape

**Tier 1 — GA text functions, runnable on the free edition.** Each gets a simple, executable example below, **in this order**:

| # | Function | What it does |
| --- | --- | --- |
| 1 | `ai_analyze_sentiment` | positive / negative / neutral / mixed |
| 2 | `ai_classify` | sorts text into labels **you** define |
| 3 | `ai_extract` | pulls named fields out of free text |
| 4 | `ai_translate` | translates text |
| 5 | `ai_mask` | masks entities (names, phones, emails, …) |
| 6 | `ai_summarize` | summarizes long text |
| 7 | `ai_gen` | answers any free-form prompt |
| 8 | `ai_fix_grammar` | corrects grammar |
| 9 | `ai_similarity` | semantic similarity score (0–1) between two texts |

**Tier 2 — general-purpose (GA).** Pattern shown, not run here (needs a Model Serving endpoint):

| Function | What it does |
| --- | --- |
| `ai_query` | calls any Model Serving endpoint with a custom prompt + output schema |

**Tier 3 — Beta / preview / heavier setup.** At the end of this notebook; runnable only where the feature is enabled — no live code except a commented `ai_decide` example:

| Function | Status | What it does |
| --- | --- | --- |
| `ai_decide` | Beta | structured YES/NO probability, option choice, or rubric score |
| `ai_enrich` | Beta | generate new columns per row from a schema you provide |
| `ai_search` / `ai_prep_search` | Beta | RAG-style search over knowledge sources |
| `ai_transcribe` | Beta | audio file to text |
| `ai_parse_document` | GA, needs a document source | parses PDFs/images/Office docs into structured content |
| `ai_forecast`, `ai_detect_anomalies`, `ai_predict_class`, `ai_predict_value`, `ai_top_drivers` | Preview (admin must enable *Predictive AI Functions*) | forecasting & classic ML from SQL |

(`ai_generate_text` is deprecated — use `ai_query` or the task-specific functions instead.)

This notebook runs the **9 GA text functions live** (tier 1, in the order above); `ai_query` is shown as a pattern; the Beta tier follows at the end.

### 1) `ai_analyze_sentiment`

**Definition:** scores the emotional tone of any text.

**Signature:** `ai_analyze_sentiment(content) → STRING`

**Parameters:**
* `content` — the text to score

**Returns:** `'positive'`, `'negative'`, `'neutral'` or `'mixed'`.

**Used in:** Notebook 02 — the `sentiment` column in `silver_delivery_reviews_enriched`, and the escalation-queue filter (`sentiment = 'negative'`).

In [0]:
%sql
SELECT
  ai_analyze_sentiment('Amazing biryani, still hot and fresh!') AS positive_example,
  ai_analyze_sentiment('Cold, late, and my order was wrong.')  AS negative_example;

### 2) `ai_classify`

**Definition:** sorts text into exactly one category, from labels you define.

**Signature:** `ai_classify(content, labels) → STRING`

**Parameters:**
* `content` — the text to classify
* `labels` — an ARRAY of your categories; the model picks one

**Prompt-design tip:** short, non-overlapping, meaningful labels give the best results.

**Used in:** Notebook 02 — the `issue_type` column (delivery_delay, food_quality, …), which routes each review to the right ops team.

In [0]:
%sql
SELECT ai_classify(
  'My order never showed up and the app charged me twice',
  ARRAY('delivery problem', 'food quality', 'app issue', 'pricing')
) AS category;

### 3) `ai_extract`

**Definition:** pulls named fields out of free text as structured JSON.

**Signature:** `ai_extract(content, schema [, options]) → VARIANT`

**Parameters:**
* `content` — raw text (or parsed-document output)
* `schema` — a JSON string listing the fields to pull out: plain names, or typed and nested schemas
* `options` — optional; `MAP('version','2.1')` enables the improved response shape (used in the example below)

Read fields with `:` path syntax — notebook 02 shows that.

**Used in:** Notebook 02 — the `food_item` column (`:response.food_item.value::STRING`).

In [0]:
%sql
SELECT ai_extract(
  'Priya ordered 2 biryanis and a coke from the Austin store',
  '["customer_name", "food_item", "city"]',
  MAP('version', '2.1')
) AS extracted;

### 4) `ai_translate`

**Definition:** translates text into a target language.

**Signature:** `ai_translate(content, target_language) → STRING`

**Parameters:**
* `content` — the text to translate
* `target_language` — a language code (`'en'`, `'es'`) or a name (`'English'`)

**Used in:** Notebook 02 — the `review_en` column: every multilingual review becomes English, so every analyst can read it.

In [0]:
%sql
SELECT
  ai_translate('Das Essen war köstlich, aber viel zu spät!', 'en') AS to_english,
  ai_translate('The food was delicious!', 'es')                    AS to_spanish;

### 5) `ai_mask`

**Definition:** detects entities and replaces each with `[MASKED]` — customer identities never reach a model.

**Signature:** `ai_mask(content, labels) → STRING`

**Parameters:**
* `content` — the text to sanitize
* `labels` — entity types to hide: `'person name'`, `'phone number'`, `'email address'`, `'address'`, `'organization'`, …

Run this **before** sending customer text to any other AI function.

**Used in:** Notebook 02 — the PII-masking step that builds `silver_delivery_reviews_masked`.

In [0]:
%sql
SELECT ai_mask(
  'Call me at 555-123-4567, my name is Priya, or email priya@example.com',
  ARRAY('phone number', 'person name', 'email address')
) AS masked;

### 6) `ai_summarize`

**Definition:** condenses long text into a short summary.

**Signature:** `ai_summarize(content [, max_words]) → STRING`

**Parameters:**
* `content` — the text to summarize
* `max_words` — optional cap on the summary length

**Used in:** not in the notebook-02 pipeline; typical use: shrinking long reviews or complaints for dashboards and support tickets.

In [0]:
%sql
SELECT ai_summarize(
  'I ordered at 7pm and the food finally arrived at 9pm, cold and squashed. The rider was polite and apologized, but this is the third time this month. The food here is usually great, but delivery needs fixing. I would order again if it arrived on time.',
  20
) AS summary;

### 7) `ai_gen`

**Definition:** free-form generation — answers any prompt you write.

**Signature:** `ai_gen(content) → STRING`

**Parameters:**
* `content` — your prompt; your one chance to *engineer* it: be specific about tone, length, constraints, and what NOT to do

**Used in:** Notebook 02 — the AI-drafted apology replies in `gold_support_escalation_queue` (on 5 rows only — cost control).

In [0]:
%sql
SELECT ai_gen(
  'Write a one-line friendly apology to a food-delivery customer whose order arrived cold. Do not promise a refund.'
) AS apology;

### 8) `ai_fix_grammar`

**Definition:** corrects grammar and spelling while preserving the meaning.

**Signature:** `ai_fix_grammar(content) → STRING`

**Parameters:**
* `content` — the text to clean up

**Used in:** not in the notebook-02 pipeline; typical use: cleaning up messy customer text before display or further processing.

In [0]:
%sql
SELECT ai_fix_grammar('this foods was cold when it arrive at mine home') AS corrected;

### 9) `ai_similarity`

**Definition:** scores how semantically similar two texts are.

**Signature:** `ai_similarity(strExpr1, strExpr2) → DOUBLE`

**Parameters:**
* `strExpr1`, `strExpr2` — the two texts to compare

**Returns:** a score between 0.0 and 1.0 — great for fuzzy deduplication and record matching.

**Used in:** not in the notebook-02 pipeline (it uses a cheap `sha2` hash for exact duplicates); typical use: near-duplicate detection at scale.

In [0]:
%sql
SELECT
  ai_similarity('The food arrived cold', 'My meal came cold') AS similar_pair,
  ai_similarity('The food arrived cold', 'Best pizza ever!')  AS different_pair;

### 10) `ai_query` — the general-purpose escape hatch (GA, pattern only)

**Definition:** invokes any Model Serving endpoint with a custom prompt — full control over model, prompt and structured output.

**Signature:** `ai_query(endpoint, request [, returnType]) → returnType`

When no task-specific function fits — custom prompts, your own model endpoint, structured output:

```sql
SELECT ai_query(
  'system.ai.meta-llama-3-3-70b-instruct',
  'Classify this delivery review. Review: ' || review_text,
  responseFormat => 'STRUCT<sentiment:STRING, severity:INT, needs_refund:BOOLEAN>'
) AS insight
FROM reviews;
-- then read insight.sentiment, insight.severity, ...
```

Not run here — it needs Model Serving access and costs more than the built-ins. Notebook 02 shows the full pattern in its optional section.

Rule of thumb: try a task-specific function (tier 1) first; drop to `ai_query` only when none fits.

## The Beta & preview tier (at the end, shown — not run)

### 11) `ai_decide` (Beta)

**Definition:** returns structured decisions — probability, choice, or rubric score — without writing a custom prompt.

**Signature:** `ai_decide(state, questions [, options]) → VARIANT`

**Parameters:**
* `state` — the text to make decisions about
* `questions` — a JSON object; each question has a `type`:
  * `"noul"` → probability that a statement is true (0–1)
  * `"choice"` → picks one of your options
  * `"score"` → scores against an ordered rubric (0 = first level)

**Returns:** a VARIANT with `response.answers`.

**Typical use:** triage rules (escalate? how urgent?) that would otherwise need a hand-rolled `ai_query` prompt.

**Workspace caveat:** the `ai_decide` endpoint is Beta and not enabled in every workspace — in this one it currently returns an error, so the example below is commented out. Uncomment it to try it where the endpoint is enabled.

In [0]:
%sql
-- ai_decide is Beta: the endpoint is not enabled in this workspace right now
-- (it returns: 'Endpoint ai_decide is not enabled for batch inference').
-- Uncomment to try it where it is enabled.
--
-- SELECT ai_decide(
--   'Order arrived 3 hours late and cold. The customer is asking for a refund.',
--   '{
--      "escalate": {"type": "noul", "instructions": "Should a human agent urgently follow up with this customer?"},
--      "priority":  {"type": "score", "instructions": "How urgent is this feedback?",
--                    "criteria": ["Routine", "Time-sensitive", "Critical"]}
--    }'
-- ) AS decision;

### The rest of the Beta / preview family (not run here)

Shown for awareness — runnable only where the feature is enabled on your workspace:

* `ai_parse_document` — parse PDFs/images/Office docs into structured content (GA, but needs a document source — no simple literal example)
* `ai_enrich` (Beta) — generate new columns per row from a schema you provide
* `ai_search` / `ai_prep_search` (Beta) — RAG-style search over knowledge sources
* `ai_transcribe` (Beta) — audio file to text
* `ai_forecast`, `ai_detect_anomalies`, `ai_predict_class`, `ai_predict_value`, `ai_top_drivers` (Preview) — forecasting & classic ML from SQL; a workspace admin must enable the *Predictive AI Functions* preview

Docs: [AI functions reference](https://docs.databricks.com/aws/en/sql/language-manual/)

## What does it cost? — tracking AI Function spend

**How AI functions are billed:**

* The compute that runs your query (SQL warehouse, cluster) is **always billed**.
* Task-specific functions (`ai_classify`, `ai_summarize`, `ai_translate`, …) run inference on **Databricks-managed serverless GPU** infrastructure — you never provision it, but it is billed in addition to your query compute.
* `ai_query` is billed for the Model Serving endpoint you point it at.
* `ai_forecast`, `ai_detect_anomalies`, `ai_predict_class`, `ai_predict_value`, `ai_top_drivers` run entirely on your own compute — **no separate inference cost**.

**Where the costs land** — in the `system.billing.usage` table, under one of two products:

| Product | Functions |
| --- | --- |
| `MODEL_SERVING` (offering type `BATCH_INFERENCE`) | `ai_similarity`, `ai_analyze_sentiment`, `ai_summarize`, `ai_translate`, `ai_fix_grammar`, `ai_mask` |
| `AI_FUNCTIONS` | the other task-specific functions, incl. `ai_classify`, `ai_extract`, `ai_parse_document`, `ai_search` |

The two cells below query the billing tables directly — run them if you have access to `system.billing` (account level):

1. **Raw spend rows** — every AI Functions usage record, with commented lines to narrow by product or single function
2. **Cost by user, monthly** — list-cost totals for both products, last 13 months

Docs: [View costs for AI Function workloads](https://docs.databricks.com/aws/en/large-language-models/ai-functions#-view-costs-for-ai-function-workloads)

In [0]:
%sql
-- 1) RAW SPEND ROWS — every AI Functions usage record, newest first.
-- Needs access to the system.billing tables (account level).
-- Uncomment one of these to narrow the view:
--   MODEL_SERVING product (ai_similarity, sentiment, summarize, translate, ...):
--     AND u.product_features.model_serving.offering_type = 'BATCH_INFERENCE'
--   a single function:
--     AND u.product_features.ai_functions.ai_function = 'AI_CLASSIFY'
SELECT *
FROM   system.billing.usage u
WHERE  u.billing_origin_product = 'AI_FUNCTIONS'
ORDER  BY u.usage_date DESC;

In [0]:
%sql
-- 2) COST BY USER, MONTHLY — list-cost totals for BOTH AI products, last 13 months.
--   list_prices:    effective list price per SKU -> usage x price = cost
--   COALESCE(run_as, run_by): attributes spend to the identity that ran the query
--   GROUP BY month x user, newest (and priciest) first
SELECT DATE_TRUNC('MONTH', u.usage_date)::DATE AS usage_month,
       COALESCE(u.identity_metadata.run_as, u.identity_metadata.run_by, 'UNATTRIBUTED') AS principal,
       ROUND(SUM(u.usage_quantity * p.pricing.effective_list.default), 2) AS list_cost_usd
FROM   system.billing.usage u
JOIN   system.billing.list_prices p
  ON   u.sku_name = p.sku_name AND u.usage_unit = p.usage_unit
   AND u.usage_end_time >= p.price_start_time
   AND (p.price_end_time IS NULL OR u.usage_end_time < p.price_end_time)
WHERE  u.usage_date >= ADD_MONTHS(DATE_TRUNC('MONTH', CURRENT_DATE()), -12)
  AND ( (u.billing_origin_product = 'MODEL_SERVING'
          AND u.product_features.model_serving.offering_type = 'BATCH_INFERENCE')
      OR u.billing_origin_product = 'AI_FUNCTIONS' )
GROUP  BY 1, 2
ORDER  BY 1 DESC, 3 DESC;

### This notebook's cost

Every example above calls an AI function on a **single literal string** — a few cents total. The production pattern is notebook 02's: call AI functions **once**, store the results in tables, then query the stored columns for free.

## Recap

| Pattern | Rule of thumb |
| --- | --- |
| Task-specific first | `ai_classify` / `ai_extract` beat a custom `ai_query` prompt when they fit |
| Cost | call AI functions on **single literals** when learning; **once, into tables** in production |
| Quality | precise labels (classify), good schemas (extract), explicit constraints (gen) |

**Next:** open `02_reviews_pipeline` — put these functions to work in a real medallion pipeline.